    # Pregunta 3 — Fauna silvestre con cámaras trampa (YOLO), paso a paso

    Las cámaras trampa se activan por movimiento y generan muchas más
    imágenes de las que un equipo humano puede revisar. Hay que detectar y
    clasificar la especie, con tres dificultades que no son las de un
    detector cualquiera:

    - **Desbalance extremo**: la mayoría de las fotos no tiene ningún animal,
      y entre las que sí, unas especies son cientos de veces más raras.
    - **Oclusión y animales pequeños o lejanos**.
    - **Generalización geográfica severa**: un modelo tiende a memorizar el
      fondo de cada cámara —una roca, un tronco, un patrón de iluminación— en
      vez de aprender rasgos del animal.


> **Cómo usar este cuaderno.** Las celdas de código son **seguras de correr**:
> inspeccionan los datos y los resultados que ya están en disco, pero
> **ninguna reentrena nada**. Los comandos que sí entrenan están escritos como
> texto, no como celdas, para que no se disparen por accidente al hacer
> "ejecutar todo".
>
> Si una celda dice que falta un archivo, es que ese paso todavía no corrió:
> el README del paso correspondiente tiene el comando exacto.

In [ ]:
import json
import pathlib
import sys

# La raiz del proyecto se busca hacia arriba, asi el cuaderno funciona tanto
# si se abre desde la carpeta de la pregunta como desde la raiz.
# Raiz del proyecto en la maquina local. Orden: la variable REDES_RAIZ, la
# carpeta actual o alguna superior, y por ultimo la ruta fija de esta PC. Asi
# funciona en Jupyter/VS Code y en Colab conectado a un entorno de ejecucion
# local, aunque Jupyter se haya lanzado desde otra carpeta.
import os
RAIZ_LOCAL = pathlib.Path("D:/Backend/redesneuronales")
RAIZ = pathlib.Path(os.environ.get("REDES_RAIZ", pathlib.Path.cwd()))
while not (RAIZ / "pregunta1").is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
if not (RAIZ / "pregunta1").is_dir():
    RAIZ = RAIZ_LOCAL
if not (RAIZ / "pregunta1").is_dir():
    raise FileNotFoundError(
        f"No encuentro el proyecto (busque desde {pathlib.Path.cwd()} y en "
        f"{RAIZ_LOCAL}). En Colab: Conectar -> 'Conectar a un entorno de "
        "ejecucion local', o define REDES_RAIZ con la ruta del proyecto.")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("raiz del proyecto:", RAIZ)


def leer(ruta):
    """Lee un JSON de resultados; avisa con claridad si todavia no existe.

    Acepta una ruta relativa a la raiz o una absoluta (la que da rutas.py).
    """
    p = pathlib.Path(ruta)
    if not p.is_absolute():
        p = RAIZ / p
    if not p.is_file():
        print(f"  (falta {ruta}: ese paso todavia no corrio)")
        return None
    return json.loads(p.read_text(encoding="utf-8"))

In [ ]:
# Que hay en disco.
p1 = RAIZ / "pregunta3/paso1_datos"
p2 = RAIZ / "pregunta3/paso2_preprocesamiento/dataset"
print(f"imagenes : {len(list((p1/'images').iterdir())) if (p1/'images').is_dir() else 0:,}")
print(f"etiquetas: {len(list((p1/'labels').glob('*.txt'))) if (p1/'labels').is_dir() else 0:,}")
if (p2 / "resumen.json").is_file():
    r = json.loads((p2 / "resumen.json").read_text(encoding="utf-8"))
    print("\nparticiones:")
    for k, v in r.items():
        if isinstance(v, dict) and "imagenes" in v:
            print(f"  {k:<12} {v['imagenes']:>7,} imagenes "
                  f"{v.get('cajas', 0):>7,} cajas")

# Pasos 1 y 2 — Los datos y su conversión

El dataset es **Caltech Camera Traps (CCT-20)**, vía LILA BC. Trae su
propia partición, y esa partición es el experimento del entregable 3:

| partición | qué es |
|---|---|
| `train`, `cis_val`, `cis_test` | **las mismas 10 cámaras** que vio el entrenamiento |
| `trans_val`, `trans_test` | **cámaras que el modelo nunca vio** |

"cis" y "trans" no son nombres decorativos: son el control y el
tratamiento de un experimento sobre generalización geográfica.

In [ ]:
p2 = RAIZ / "pregunta3/paso2_preprocesamiento/dataset"
if (p2 / "resumen.json").is_file():
    r = json.loads((p2 / "resumen.json").read_text(encoding="utf-8"))
    for k, v in r.items():
        if isinstance(v, dict):
            extra = ""
            if v.get("por_fuga_de_rafaga"):
                extra = (f"  <- {v['por_fuga_de_rafaga']} descartadas por "
                         f"FUGA DE RAFAGA")
            print(f"{k:<12} {v.get('imagenes',0):>7,} img "
                  f"{v.get('cajas',0):>7,} cajas{extra}")
    print("\n  La fuga de rafaga es el detalle fino: las camaras disparan")
    print("  en rafagas de 3 fotos casi identicas. Si una cae en train y")
    print("  otra en validacion, la validacion mide memorizacion.")

### Las imágenes **no se copian** al armar el dataset

Son 18 GB. Las listas de Ultralytics apuntan a `paso1_datos/images/` y las
etiquetas viven en `paso1_datos/labels/`, que es donde YOLO las busca por
convención (reemplaza `/images/` por `/labels/` en la ruta).

# Paso 3 — El modelo (entregable 1)

El enunciado pide documentar **tres** decisiones: resolución de entrada,
estrategia de aumento de datos, y qué se hace con las imágenes vacías.

In [ ]:
r = leer("pregunta3/paso3_modelo/corridas/yolo11n_640/resumen.json")
if r:
    for k in ("map50", "map50_95", "precision", "recall",
              "epocas_en_results_csv", "procesos", "reanudado",
              "minutos_totales_aproximados"):
        if k in r:
            v = r[k]
            print(f"  {k:<28} {v:.5f}" if isinstance(v, float)
                  else f"  {k:<28} {v}")

### Un detalle que cambia cómo se lee ese 0.9342

Ese número es el **pico de una época**, no la meseta del final del
entrenamiento. Conviene verlo en la curva antes de citarlo.

In [ ]:
import csv as _csv

ruta = RAIZ / "pregunta3/paso3_modelo/corridas/yolo11n_640/results.csv"
if ruta.is_file():
    with ruta.open() as f:
        filas = list(_csv.DictReader(f))
    col = {k.strip(): k for k in filas[0]}
    ep = [int(float(x[col["epoch"]])) for x in filas]
    m50 = [float(x[col["metrics/mAP50(B)"]]) for x in filas]
    mejor = max(range(len(m50)), key=lambda i: m50[i])
    print(f"maximo    : epoca {ep[mejor]} -> mAP50 {m50[mejor]:.5f}  (es best.pt)")
    print(f"ultima    : epoca {ep[-1]} -> mAP50 {m50[-1]:.5f}")
    print(f"diferencia: {m50[mejor]-m50[-1]:+.4f}")
    print("\n  Citar 0.9342 sin decir que es un pico lo hace parecer")
    print("  elegido a dedo. La meseta final esta ~0.08 por debajo.")

In [ ]:
if ruta.is_file():
    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                         "grid.alpha": 0.3, "font.size": 9})
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    axes[0].plot(ep, m50, label="mAP@0.5")
    axes[0].plot(ep, [float(x[col["metrics/mAP50-95(B)"]]) for x in filas],
                 label="mAP@0.5:0.95")
    axes[0].axvline(ep[mejor], ls="--", c="crimson", lw=1,
                    label=f"best.pt (ep {ep[mejor]})")
    axes[0].axvline(50, ls=":", c="gray", lw=1, label="close_mosaic")
    axes[0].set_xlabel("epoca"); axes[0].legend(fontsize=7)
    axes[0].set_title("Metricas de validacion")

    for c, et in (("train/box_loss", "box"), ("train/cls_loss", "cls"),
                  ("train/dfl_loss", "dfl")):
        axes[1].plot(ep, [float(x[col[c]]) for x in filas], label=et)
    axes[1].axvline(50, ls=":", c="gray", lw=1)
    axes[1].set_xlabel("epoca"); axes[1].legend(fontsize=7)
    axes[1].set_title("Perdidas de entrenamiento")
    plt.tight_layout(); plt.show()

    i = ep.index(50)
    print("en la epoca 51 se apaga el mosaico (close_mosaic=10):")
    for c in ("train/box_loss", "train/cls_loss", "train/dfl_loss"):
        a, b = float(filas[i][col[c]]), float(filas[i+1][col[c]])
        print(f"  {c:<16} {a:.5f} -> {b:.5f}  {'BAJA' if b < a else 'sube'}")
    print(f"  {'mAP50':<16} {m50[i]:.5f} -> {m50[i+1]:.5f}")
    print(f"  y 3 epocas despues: {m50[i+4]:.5f}  (+{m50[i+4]-m50[i]:.4f})")
    print("\n  La que baja es la perdida de CLASIFICACION. Las de caja (box,")
    print("  dfl) suben. Y el mAP SI pega el mayor salto de toda la corrida.")

# Paso 4 — Evaluación (entregable 2)

Las métricas se calculan **acá**, no se delegan a Ultralytics, por una
razón: para poder poner un intervalo de confianza alrededor. Y después se
contrastan con las de Ultralytics como control de que la implementación
no tiene un error.

In [ ]:
m = leer("pregunta3/paso4_evaluacion/informes/metricas.json")
if m:
    print(f"{'particion':<12} {'img':>7} {'cajas':>7} {'mAP50':>8} "
          f"{'IC95%':>20} {'mAP50-95':>10} {'ponder':>8}")
    print("-" * 80)
    for k, v in m.items():
        if not isinstance(v, dict) or "map50" not in v:
            continue
        ic = (v.get("bootstrap") or {}).get("ic95", [float("nan")]*2)
        print(f"{k:<12} {v.get('imagenes',0):>7,} {v.get('cajas',0):>7,} "
              f"{v['map50']:>8.4f}  [{ic[0]:.4f},{ic[1]:.4f}] "
              f"{v.get('map50_95',0):>10.4f} "
              f"{v.get('map50_ponderado',0):>8.4f}")

In [ ]:
c = leer("pregunta3/paso4_evaluacion/informes/contraste_ultralytics.json")
if c:
    print("control: nuestras metricas contra las de Ultralytics\n")
    for k, v in c.items():
        if isinstance(v, dict) and "diferencia" in v:
            print(f"  {k:<12} propia {v.get('map50_propio', 0):.4f} | "
                  f"ultralytics {v.get('ultralytics_map50', 0):.4f} | "
                  f"dif {v['diferencia']:+.4f}")
    print("\n  Las diferencias son chicas y del mismo signo en las dos")
    print("  particiones: la implementacion propia es consistente, apenas")
    print("  mas conservadora. La BRECHA entre particiones -que es lo que")
    print("  mide el entregable 3- sale casi igual por los dos caminos.")

# Paso 5 — Generalización y robustez (entregables 3 y 4)

## El resultado central

In [ ]:
g = leer("pregunta3/paso5_generalizacion/informes/generalizacion.json")
if g:
    print(f"brecha mAP50 (cis - trans): {g['brecha_map50']:+.4f}")
    print(f"  intervalos disjuntos: {g.get('intervalos_disjuntos')}")
    co = g.get("controlado", {})
    if co:
        print(f"\ncontrolando la mezcla de especies ({co.get('imagenes',0):,} img):")
        print(f"  mAP50 controlado       : {co.get('map50',0):.4f}")
        print(f"  brecha controlada      : {g.get('brecha_map50_controlada',0):+.4f}")
        resto = g['brecha_map50'] - g.get('brecha_map50_controlada', 0)
        print(f"\n  la mezcla de especies explica {resto:.4f} de la brecha;")
        print(f"  el {100*g.get('brecha_map50_controlada',0)/g['brecha_map50']:.0f}% "
              f"restante es atribuible a la UBICACION.")
    for k, et in (("dispersion_vista", "cis (vistas)"),
                  ("dispersion_nueva", "trans (nuevas)")):
        d = g.get(k, {})
        if d:
            # Las claves del JSON son 'minimo'/'maximo'/'desviacion'.
            # Se leen del archivo, no se adivinan por el nombre corto.
            print(f"\n{et}: {d['n']} camaras | media {d['media']:.4f} "
                  f"| min {d['minimo']:.4f} | max {d['maximo']:.4f} "
                  f"| desviacion {d['desviacion']:.4f}")

### Por qué el control de distribución es lo que hace creíble el resultado

Las cámaras nuevas no solo son nuevas: también tienen **otra mezcla de
especies**. Sin controlar eso, la brecha podría ser simplemente "hay más
de las especies difíciles". Remuestreando `trans_test` para que su mezcla
se parezca a la de `cis_test`, la brecha casi no baja — así que **no es la
mezcla, es la ubicación**.

In [ ]:
r = leer("pregunta3/paso5_generalizacion/informes/robustez.json")
if r:
    oc = r.get("oclusion", {})
    if oc:
        print("OCLUSION sintetica (parche gris; el modelo nunca la vio)\n")
        fracciones = sorted(oc.get("centro", {}))
        print(f"{'fraccion tapada':>16} {'centro':>9} {'inferior':>10}")
        print("-" * 38)
        for k in fracciones:
            c = oc["centro"][k]["map50"]
            i = oc.get("inferior", {}).get(k, {}).get("map50", float("nan"))
            print(f"{k:>16} {c:>9.4f} {i:>10.4f}")
        base = oc["centro"][fracciones[0]]["map50"]
        if len(fracciones) > 1:
            cae = oc["centro"][fracciones[1]]["map50"]
            print(f"\n  tapar el {fracciones[1]} del area cuesta el "
                  f"{100*(base-cae)/base:.0f}% del mAP.")
            print("  La oclusion en el CENTRO duele mas que abajo, que es lo")
            print("  contrario de lo que uno disenaria esperando (las patas")
            print("  tapadas por vegetacion parecen el caso tipico).")

    tam = r.get("tamano", {})
    if tam:
        print("\n\nTAMANO del objeto (fraccion del cuadro)\n")
        print(f"{'tramo':<12} {'cis_test':>10} {'trans_test':>12} "
              f"{'cajas cis':>11}")
        print("-" * 48)
        porc = {p: {x["tramo"]: x for x in v} for p, v in tam.items()}
        for t_ in ("diminuto", "chico", "mediano", "grande"):
            a = porc.get("cis_test", {}).get(t_, {})
            b = porc.get("trans_test", {}).get(t_, {})
            if not a:
                continue
            print(f"{t_:<12} {a['map50']:>10.4f} "
                  f"{b.get('map50', float('nan')):>12.4f} "
                  f"{a.get('cajas', 0):>11,}")
        print("\n  El tamano explica parte de la dificultad, pero NO la")
        print("  brecha: en todos los tramos trans cae respecto de cis.")

# Paso 6 — Monitoreo ecológico (entregable 5)

Una detección no es un animal: una ráfaga de 3 fotos del mismo zorro son
3 detecciones y **un** avistamiento. La unidad de la ecología es el
**evento independiente**: misma especie, misma cámara, separados por más
de un intervalo de silencio (convención: 30 minutos).

In [ ]:
a = leer("pregunta3/paso6_monitoreo/informes/actividad.json")
if a:
    print("parametros:", a.get("parametros"))
    print("esfuerzo  :", a.get("esfuerzo"))
    print("cobertura :", a.get("cobertura"))
    esp = a.get("especies", {})
    if isinstance(esp, dict):
        esp = [{"especie": k, **v} for k, v in esp.items()]
    det = sum(x.get("detecciones", 0) for x in esp)
    ev = sum(x.get("eventos", 0) for x in esp)
    print(f"\n{det:,} detecciones -> {ev:,} eventos independientes "
          f"(x{det/max(1,ev):.1f})")
    print(f"\n{'especie':<14} {'eventos':>8} {'anotados':>9} {'error':>8} "
          f"{'nocturnidad':>12}")
    print("-" * 56)
    for x in sorted(esp, key=lambda z: -z.get("eventos", 0))[:8]:
        err = x.get("error_relativo", x.get("error", 0))
        print(f"{x.get('especie','?'):<14} {x.get('eventos',0):>8} "
              f"{x.get('anotados',0):>9} {err:>7.0%} "
              f"{x.get('nocturnidad',0):>11.0%}")

### El control de sanidad que vale más que la métrica

El índice de nocturnidad —qué fracción de los eventos cae entre las 20:00
y las 05:00— **reproduce la historia natural de las especies detectadas**
sin que nadie se lo haya dicho al modelo. Los diurnos dan cerca de 0, los
nocturnos cerca de 0.8, los crepusculares en el medio.

Que eso salga solo es la señal más fuerte de que las detecciones no son
ruido: un detector que se equivocara al azar daría nocturnidad uniforme.

# Resumen de los seis entregables

| # | qué pide | dónde |
|---|---|---|
| 1 | YOLO ajustado, con las 3 decisiones documentadas | paso 3 |
| 2 | mAP@0.5 y mAP@0.5:0.95 con desglose por especie | paso 4 |
| 3 | generalización entre ubicaciones + Grad-CAM | paso 5 |
| 4 | robustez ante oclusión y tamaño | paso 5 |
| 5 | módulo de monitoreo ecológico | paso 6 |
| 6 | media página sobre IA responsable en conservación | paso 7 |

# Las tres lecciones

**1. La partición ya era el experimento.** `cis` contra `trans` no es un
reparto administrativo: es el control y el tratamiento. Entenderlo cambia
qué se puede concluir.

**2. Una brecha sin control de confundidores no prueba nada.** Las cámaras
nuevas tienen otra mezcla de especies; recién después de controlarla se
puede atribuir la caída a la ubicación.

**3. Un detector no es un sistema de monitoreo.** De detecciones a eventos
independientes, y de ahí a tasas por esfuerzo de cámara, hay un trabajo
que no es de visión por computadora y sin el cual el número no significa
nada para un ecólogo.